# Vector Database con ChromaDB


## Importación de librerias necesarias


In [17]:
%load_ext IPython.extensions.autoreload
%autoreload 2


The IPython.extensions.autoreload extension is already loaded. To reload it, use:
  %reload_ext IPython.extensions.autoreload


In [18]:
import os
from pathlib import Path
import sys

def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path

src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))


In [19]:
import numpy as np
import pandas as pd
from pyspark.sql import Window, functions as F
from src.utils.chromadb_connector import ChromaDBConnector
from src.utils.spark import SparkUtils


In [20]:
spark_utils = SparkUtils('chromadb_vectors')
spark = spark_utils.spark


## Configurar ChromaDB


In [21]:
COLLECTION_NAME = "item_embeddings"
chroma_connector = ChromaDBConnector(collection_name=COLLECTION_NAME)


## Cargar embeddings desde Delta


In [27]:
GOLD_SCHEMA_ENCODING = 'gold.encoding'
meta_items_title_text_clean_embeddings = (
    spark.read.format('parquet').load(spark_utils.path(
        'meta_items_title_text_clean_embeddings', 
        catalog = GOLD_SCHEMA_ENCODING
    ))
).withColumn('component', F.lit('title'))
meta_items_description_sentences_text_clean_embeddings = (
    spark.read.format('parquet').load(spark_utils.path(
        'meta_items_description_sentences_text_clean_embeddings', 
        catalog = GOLD_SCHEMA_ENCODING
    ))
).withColumn('component', F.lit('description'))
meta_items_features_text_clean_embeddings = (
    spark.read.format('parquet').load(spark_utils.path(
        'meta_items_features_text_clean_embeddings', 
        catalog = GOLD_SCHEMA_ENCODING
    ))
).withColumn('component', F.lit('features'))


In [28]:
meta_items_encodings = meta_items_title_text_clean_embeddings.union(
    meta_items_description_sentences_text_clean_embeddings
).union(
    meta_items_features_text_clean_embeddings
)

In [29]:
meta_items_encodings.count()

7492943

In [ ]:
LOAD_TO_CHROMADB = True

## Agregar vectores a ChromaDB


In [ ]:
if LOAD_TO_CHROMADB:
    batch_size = 1_000
    w = Window.orderBy('parent_asin', 'component')
    df_idx = meta_items_encodings.withColumn("rn", F.row_number().over(w))
    total_count = df_idx.count()
    print(f"Total records: {total_count}")
    
    offset = 0
    batch_idx = 0
    
    while offset < total_count:
        batch_df = df_idx.filter(
            (F.col("rn") > offset) & (F.col("rn") <= offset + batch_size)
        ).select('parent_asin', 'sum_embedding')
        
        pdf = batch_df.toPandas()
        
        ids = pdf['parent_asin'].astype(str).tolist()
        embeddings = [list(emb) for emb in pdf['sum_embedding'].tolist()]
        metadatas = [{'parent_asin': asin} for asin in ids]
        
        chroma_connector.add_vectors(
            ids=ids,
            embeddings=embeddings,
            metadatas=metadatas
        )
        
        offset += batch_size
        batch_idx += 1
        print(f"Processed batch {batch_idx}, offset: {offset}")
    
    print(f"Collection count: {chroma_connector.count()}")


Total records: 456530


Processed batch 1, offset: 1000


Processed batch 2, offset: 2000


ERROR:root:KeyboardInterrupt while sending command.                 (0 + 1) / 1]
Traceback (most recent call last):
  File "/mnt/d/Maestría/Amazon Reviews Code/.venv-linux/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/d/Maestría/Amazon Reviews Code/.venv-linux/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

Exception in thread "serve-DataFrame" java.net.SocketTimeoutException: Accept timed out
	at java.base/java.net.PlainSocketImpl.socketAccept(Native Method)
	at java.base/java.net.AbstractPlainSocketImpl.accept(AbstractPlainSocketImpl.java:474)
	at java.base/java.net.ServerSocket.implAccept(ServerSocket.java:565)
	at java.base/java.net.ServerSocket.accept(ServerSocket.java:533)
	at org.apache.spark.security.SocketAuthServer$$anon$1.run(SocketAuthServer.scala:65)


## Búsqueda por similitud


In [ ]:
query_asin = "1610121147"
query_item = meta_items_embeddings.filter(
    F.col('parent_asin') == query_asin
).limit(1).toPandas()

if len(query_item) > 0:
    query_embedding = query_item['sum_embedding'].iloc[0]
    query_vector = list(query_embedding) if isinstance(query_embedding, np.ndarray) else query_embedding
    
    results = chroma_connector.similarity_search(
        query_embeddings=query_vector,
        n_results=10
    )
    
    print(f"Found {len(results['ids'][0])} similar items")
    for i, (item_id, distance) in enumerate(zip(results['ids'][0], results['distances'][0])):
        print(f"{i+1}. ASIN: {item_id}, Distance: {distance:.4f}")


Found 0 similar items


## Búsqueda por similitud en batch


In [ ]:
sample_size = 100
sample_items = meta_items_embeddings.limit(sample_size).toPandas()

query_embeddings = [list(emb) if isinstance(emb, np.ndarray) else emb 
                    for emb in sample_items['sum_embedding'].tolist()]
query_ids = sample_items['parent_asin'].astype(str).tolist()

batch_results = chroma_connector.similarity_search_batch(
    query_embeddings=query_embeddings,
    n_results=5,
    batch_size=50
)

print(f"Processed {len(query_ids)} queries")
print(f"Results shape: {len(batch_results['ids'])} queries, {len(batch_results['ids'][0])} results per query")


Processed 100 queries
Results shape: 100 queries, 0 results per query


## Información de la colección


In [ ]:
info = chroma_connector.get_collection_info()
print(f"Collection: {info['name']}")
print(f"Total vectors: {info['count']}")


Collection: item_embeddings
Total vectors: 0
